# EDA - Beijing Multi-Site Air Quality

Analisis exploratorio del **caso guia**: 12 estaciones de monitoreo en Beijing,
2013-03 a 2017-02, lecturas horarias de contaminantes y meteorologia.

- **Target** (regresion): `PM2.5` (ug/m3).
- **Particiones fijas**: train / valid / test / produccion.


## 1. Configuracion e imports


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from BeijingAir.config import COL_TIEMPO
from BeijingAir.data.descarga import cargar_crudo

plt.rcParams["figure.figsize"] = (10, 5)
sns.set_theme(style="whitegrid")


## 2. Carga del crudo

Se trabaja sobre el **crudo** (tal como llega del proveedor) para entender el dato:
nulos reales, rangos, tipos.


In [ ]:
df = cargar_crudo()
print("Shape:", df.shape)
print("Rango temporal:", df[COL_TIEMPO].min(), "a", df[COL_TIEMPO].max())
print("\nTipos de dato:")
print(df.dtypes)
df.head()


## 3. Eje temporal

Frecuencia horaria, un registro por estacion y hora. Cuenta las filas por mes:
aparece el ciclo anual y la cobertura de cada tramo.

In [ ]:
df["anio_mes"] = df[COL_TIEMPO].dt.to_period("M")
df["anio_mes"].value_counts().sort_index().plot(
    kind="bar", figsize=(14, 4), title="Registros por mes"
)
plt.show()


## 4. Cobertura por estacion

Las 12 estaciones no registran la misma cantidad de horas; la cobertura es heterogenea.


In [ ]:
print("Estaciones:", df["station"].nunique())
df["station"].value_counts().plot(kind="bar", figsize=(10, 4), title="Registros por estacion")
plt.show()


## 5. Nulos por columna

En este dataset los nulos **no son estructurales**: el sensor no reporto esa hora
(fallo de captura), no que la magnitud no exista.

In [ ]:
nulos = df.isna().sum()
nulos_pct = (df.isna().mean() * 100).round(2)
tabla_nulos = pd.DataFrame({"nulos": nulos, "%": nulos_pct})
tabla_nulos = tabla_nulos[tabla_nulos["nulos"] > 0].sort_values("nulos", ascending=False)
tabla_nulos


In [ ]:
tabla_nulos["nulos"].plot(kind="barh", figsize=(8, 5), title="Nulos por columna")
plt.xlabel("nulos")
plt.show()


## 6. Distribucion de contaminantes

Concentraciones muy sesgadas a la derecha; por eso los histogramas van en escala
logaritmica. PM2.5 y PM10 se saturan en el cap del instrumento (999).

In [ ]:
contaminantes = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), contaminantes):
    sns.histplot(df[col].dropna(), bins=60, ax=ax, log_scale=True)
    ax.set_title(col)
plt.tight_layout()
plt.show()


## 7. Meteorologia

Variables meteorologicas en unidades SI (TEMP y DEWP en grados C, PRES en hPa,
RAIN en mm, WSPM en m/s). Incluye una regla fisica:
**el punto de rocio nunca supera la temperatura**.

In [ ]:
meteo = ["TEMP", "PRES", "DEWP", "RAIN", "WSPM"]
fig, axes = plt.subplots(1, len(meteo), figsize=(20, 4))
for ax, col in zip(axes, meteo):
    sns.histplot(df[col].dropna(), bins=60, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()


In [ ]:
ambas = df.dropna(subset=["TEMP", "DEWP"])
print("Violaciones de TEMP < DEWP:", int((ambas["TEMP"] < ambas["DEWP"]).sum()))


## 8. Relaciones entre contaminantes

Los contaminantes de particulas suelen estar correlacionados (misma fuente de
emision).

In [ ]:
corr = df[["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlacion entre contaminantes")
plt.show()


In [ ]:
sns.scatterplot(data=df.sample(10_000, random_state=42), x="PM10", y="PM2.5", alpha=0.3)
plt.title("PM2.5 vs PM10")
plt.show()


## 9. El target PM2.5

Distribucion, saturacion y dos ciclos del dato: por estacion
(heterogeneidad) y por hora (ciclo diario de la contaminacion).

In [ ]:
print(df["PM2.5"].describe().round(1))
print("\nValores en el cap (999):", int((df["PM2.5"] == 999).sum()))

sns.histplot(df["PM2.5"].dropna(), bins=80)
plt.title("Distribucion de PM2.5")
plt.show()


In [ ]:
sns.boxplot(data=df.dropna(subset=["PM2.5"]), x="station", y="PM2.5")
plt.xticks(rotation=90)
plt.title("PM2.5 por estacion")
plt.show()


In [ ]:
df["hora"] = df[COL_TIEMPO].dt.hour
sns.lineplot(data=df.dropna(subset=["PM2.5"]), x="hora", y="PM2.5", errorbar=None)
plt.title("PM2.5 promedio por hora")
plt.show()


In [ ]:
ciclo_estacion_hora = (
    df.dropna(subset=["PM2.5"])
    .groupby(["station", "hora"])["PM2.5"]
    .mean()
    .unstack()
)
plt.figure(figsize=(14, 6))
sns.heatmap(ciclo_estacion_hora, cmap="mako")
plt.title("PM2.5 promedio por estacion y hora")
plt.xlabel("Hora")
plt.ylabel("Estacion")
plt.show()


## 10. Conclusiones

- **Nulos** = fallo de captura (1-5 %), mas en contaminantes que en meteorologia.
- **Sesgo fuerte** a la derecha en todos los contaminantes y cap en 999.
- **Correlacion alta** entre contaminantes de particulas (PM2.5/PM10/NO2).
- **Heterogeneidad** por estacion y **ciclo diario** claro; el mapa de calor muestra ambos a la vez.